In [ ]:
"""
sandbox_mf_trialhistory.ipynb

A sandbox to test if p(mf) is correlated with trial history representation.

Author: Stellina X. Ao
Created: 2026-07-02
Last Modified: 2026-07-02
Python Version: 3.11.14
"""

import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt
import numpy as np

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
from core.data import subject_ids, session_ids

subj_id = "MR82"
sess_ids = session_ids[np.where(subject_ids == subj_id)[0][0]]
subj_sess_ids = [
    (subj_id, sess_id)
    for subj_id in ["MR82", "MR83"]
    for sess_id in session_ids[np.where(subject_ids == subj_id)[0][0]]
]

In [ ]:
"""TODO"""
# check MR83
# add current regressors
# split into strategies

## p(mf)

In [ ]:
# p(mf)

from core.data import load_sess

p_mf = np.zeros(len(subj_sess_ids))

for i, (subj_id, sess_id) in enumerate(subj_sess_ids):
    _, trial_data, _, _, _ = load_sess(subj_id, sess_id)
    p_mf[i] = np.mean(trial_data["strategy"] == -1)

In [ ]:
from core.viz import plot_raincloud

plot_raincloud(p_mf, label="p(mf)")

In [ ]:
plt.figure(figsize=(1.5, 1.25), tight_layout=True)
plt.plot(p_mf)
plt.xlabel("sessions")
plt.ylabel("p(mf)")
plt.show()

## cv/delta r2

In [ ]:
# cv/d r2

from sg.models import ShuffledEncoder

regressors = ["response", "rewarded", "response_prev", "rewarded_prev"]

n_iters = 3

cvr2s = {regr: np.zeros((len(subj_sess_ids), n_iters)) for regr in regressors}
dr2s = {regr: np.zeros((len(subj_sess_ids), n_iters)) for regr in regressors}

for i, (subj_id, sess_id) in enumerate(subj_sess_ids):
    se = ShuffledEncoder(
        subj_id,
        sess_id,
        tv_keys=[
            "response",
            "rewarded",
            "block_side",
            "strategy",
            "response_prev",
            "rewarded_prev",
        ],
    )

    for regr in regressors:
        se.get_cvr2(pivot=regr, n_iters=n_iters)
        se.get_dr2(pivot=regr, n_iters=n_iters)

        cvr2s[regr][i] = se.cvr2[regr]
        dr2s[regr][i] = se.dr2[regr]

In [ ]:
from core.viz import plot_scatter


def plot_pmf_r2(mode="cv"):
    fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(4.5, 3.5), tight_layout=True)

    for i, ax in enumerate(axes.flat):
        regr = regressors[i]

        r2 = cvr2s[regr].mean(axis=1) if mode == "cv" else dr2s[regr].mean(axis=1)
        plot_scatter(p_mf, r2, xlabel="p(mf)", ylabel=rf"{mode} $r^2$ {regr}", ax=ax)


plot_pmf_r2(mode="cv")
plot_pmf_r2(mode=r"$\Delta$")

## weights

In [ ]:
# weights
from core.data import tv_vals
from sg.models import Encoder

regressors = ["response", "rewarded", "response_prev", "rewarded_prev"]
weights = {f"{regr}_{val}": [] for regr in regressors for val in tv_vals[regr]}

for subj_id, sess_id in subj_sess_ids:
    encoder = Encoder(subj_id, sess_id)
    encoder.fit_encoder()

    for regr in regressors:
        for val in tv_vals[regr]:
            weights[f"{regr}_{val}"].append(encoder.get_weights(regr=regr, val=val))

### plots

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(3.5, 1.5), tight_layout=True)

i = 0
for regr in regressors:
    if "prev" not in regr:
        ax = axes.flat[i]
        val = tv_vals[regr][0]  # symmetric for the current trial regressors
        weights_mean = [
            np.abs(weights_sess).mean() for weights_sess in weights[f"{regr}_{val}"]
        ]
        plot_scatter(
            p_mf, weights_mean, xlabel="p(mf)", ylabel=rf"$\beta$ {regr}", ax=ax
        )
        i += 1

fig, axes = plt.subplots(ncols=5, figsize=(8.5, 1.5), tight_layout=True)

i = 0
for regr in regressors:
    if "prev" in regr:
        for val in tv_vals[regr]:
            ax = axes.flat[i]
            weights_mean = [
                np.abs(weights_sess).mean() for weights_sess in weights[f"{regr}_{val}"]
            ]
            plot_scatter(
                p_mf,
                weights_mean,
                xlabel="p(mf)",
                ylabel=rf"$\beta$ {regr}_{val}",
                ax=ax,
            )
            i += 1

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3), tight_layout=True)
ax.plot(p_mf, label="p(mf)")
ax.set_xlabel("sessions")
ax.set_ylabel("p(mf)")
ax.legend()

ax2 = ax.twinx()
ax2.plot(
    [np.abs(weights_sess).mean() for weights_sess in weights[f"{regr}_{val}"]],
    color="#89C301",
    label="weights",
)
ax2.set_ylabel("weights")
ax2.legend()